# Semana 2: segundo alvo (T2), teste do WorldFloods v2

Fase 4 do doc 20. **Nenhum modelo roda neste notebook.** Tudo roda em CPU, então não precisa de GPU.

**Parte A (células 1 a 4), pode rodar já.** Baixa só o teste (cerca de 3,2 GB) na revisão fixada do Hugging Face, confere o hash de cada arquivo, copia para `Mestrado/dados/worldfloods_v2` e inspeciona bandas, escala, grade e codificação do rótulo. A inspeção não calcula fração de nuvem nem de água.

**Parte B (células 5 e 6), só depois que os limiares estiverem registrados no repositório** (`prereg/t2_criterios.md`). Calcula a fração dos pixels rotulados que cai em país do Sen1Floods11, a fração de nuvem, aplica os três critérios do doc 22, seção 10.2, e grava `t2_eventos.csv`.

Abra o Colab **logada na conta do mestrado**. Rode na ordem, uma célula de cada vez (Shift+Enter). Se alguma célula der erro, pare e cole o erro no Claude.

Saídas em `Mestrado/resultados/semana2`: `t2_arquivos_sha256.csv` (Parte A), `t2_eventos.csv` e `t2_eventos_info.json` (Parte B).

In [ ]:
# CÉLULA 1. Montar o Drive do mestrado e preparar o ambiente
!pip install -q "rasterio==1.4.3"

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, re, json, time, hashlib, shutil, datetime, platform
import numpy as np, pandas as pd
import rasterio, huggingface_hub
from rasterio.enums import Resampling

MEUDRIVE = '/content/drive/MyDrive'
MESTRADO = os.path.join(MEUDRIVE, 'Mestrado')
assert not os.path.isdir(os.path.join(MEUDRIVE, 'DOC old PC')), (
    'Este é o Drive da conta antiga. Vá em Ambiente de execução > Desconectar e excluir ambiente de execução, '
    'abra o Colab logada na conta do mestrado e rode de novo.')
assert os.path.isdir(MESTRADO), f'Não achei a pasta Mestrado. O Meu Drive desta conta tem: {sorted(os.listdir(MEUDRIVE))}'

REPO_WF = 'isp-uv-es/WorldFloodsv2'
REVISAO_WF = '1f3faa2989e69930ac31d5c0a79fd224461123f8'    # commit de 31/07/2025 do dataset no Hugging Face
LOCAL = '/content/wf2'
DESTINO = os.path.join(MESTRADO, 'dados', 'worldfloods_v2')
SAIDA = os.path.join(MESTRADO, 'resultados', 'semana2')
for p in (DESTINO, SAIDA):
    os.makedirs(p, exist_ok=True)

total, usado, livre = shutil.disk_usage(MEUDRIVE)
print(f'Drive: {livre / 1e9:.1f} GB livres. O teste ocupa cerca de 3,2 GB.')
if livre < 4e9:
    print('ATENÇÃO: pode faltar espaço no Drive. Libere espaço antes da célula 3.')

VERSOES = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
           'rasterio': rasterio.__version__, 'gdal': rasterio.__gdal_version__,
           'huggingface_hub': huggingface_hub.__version__}
print(VERSOES)

def sha256(p, bloco=1 << 24):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(bloco), b''):
            h.update(b)
    return h.hexdigest()

In [ ]:
# CÉLULA 2. Baixar só o teste, na revisão fixada, e conferir o hash de cada arquivo (5 a 10 min)
from huggingface_hub import snapshot_download, HfApi
from huggingface_hub.hf_api import RepoFile

PADROES = ['test/*', 'dataset_metadata.csv', 'README.md']
t0 = time.time()
snapshot_download(repo_id=REPO_WF, repo_type='dataset', revision=REVISAO_WF,
                  allow_patterns=PADROES, local_dir=LOCAL, max_workers=8)
print(f'Download: {(time.time() - t0) / 60:.1f} min')

def sha1_git(p, bloco=1 << 24):
    h = hashlib.sha1(f'blob {os.path.getsize(p)}\0'.encode())
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(bloco), b''):
            h.update(b)
    return h.hexdigest()

remotos = [x for x in HfApi().list_repo_tree(REPO_WF, repo_type='dataset', revision=REVISAO_WF, recursive=True)
           if isinstance(x, RepoFile) and (x.path.startswith('test/') or x.path in ('dataset_metadata.csv', 'README.md'))]

linhas = []
for x in sorted(remotos, key=lambda r: r.path):
    p = os.path.join(LOCAL, x.path)
    assert os.path.exists(p), f'Faltou baixar {x.path}. Rode a célula de novo.'
    h = sha256(p)
    if x.lfs is not None:
        esperado = x.lfs['sha256'] if isinstance(x.lfs, dict) else x.lfs.sha256
        ok = (h == esperado)
    else:
        ok = (sha1_git(p) == x.blob_id)
    linhas.append({'path': x.path, 'bytes': os.path.getsize(p), 'sha256': h, 'matches_hf': ok})

MANIFESTO = pd.DataFrame(linhas)
print(f'{len(MANIFESTO)} arquivos, {MANIFESTO.bytes.sum() / 1e9:.2f} GB')
pasta = MANIFESTO.path.map(lambda p: p.split('/')[1] if p.startswith('test/') else '(raiz)')
print(MANIFESTO.groupby(pasta).agg(arquivos=('path', 'size'), GB=('bytes', lambda b: round(b.sum() / 1e9, 3))))
ruins = MANIFESTO[~MANIFESTO.matches_hf]
assert ruins.empty, f'Hash diferente do Hugging Face em {list(ruins.path)}. Cole esta saída no Claude.'
assert len(MANIFESTO) == 92, f'Esperava 92 arquivos (5 pastas de 18 no teste, mais 2 na raiz), vieram {len(MANIFESTO)}. Cole esta saída no Claude.'
MANIFESTO.to_csv(os.path.join(SAIDA, 't2_arquivos_sha256.csv'), index=False)
print('Todos os arquivos conferem com o Hugging Face. Manifesto salvo em resultados/semana2/t2_arquivos_sha256.csv')

In [ ]:
# CÉLULA 3. Copiar para o Drive (pula o que já está lá com o mesmo tamanho) e conferir (5 a 15 min)
t0 = time.time(); copiados = 0
for r in MANIFESTO.itertuples():
    src, dst = os.path.join(LOCAL, r.path), os.path.join(DESTINO, r.path)
    if os.path.exists(dst) and os.path.getsize(dst) == r.bytes:
        continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    copiados += 1
print(f'{copiados} arquivos copiados em {(time.time() - t0) / 60:.1f} min')

faltando = [r.path for r in MANIFESTO.itertuples()
            if not os.path.exists(os.path.join(DESTINO, r.path)) or os.path.getsize(os.path.join(DESTINO, r.path)) != r.bytes]
assert not faltando, f'{len(faltando)} arquivos faltando ou incompletos no Drive, por exemplo {faltando[:3]}. Rode a célula de novo.'
MANIFESTO.to_csv(os.path.join(DESTINO, 't2_arquivos_sha256.csv'), index=False)
print(f'Cópia em {DESTINO} conferida por tamanho: {len(MANIFESTO)} arquivos.')

In [ ]:
# CÉLULA 4. Inspeção: bandas, escala, grade e codificação do rótulo
# Não calcula fração de nuvem nem de água: isso fica para a Parte B, depois do registro dos limiares.
DADOS = LOCAL if os.path.isdir(os.path.join(LOCAL, 'test', 'S2')) else DESTINO
META = pd.read_csv(os.path.join(DADOS, 'dataset_metadata.csv'))
TESTE = META[META['split'] == 'test'].sort_values('event id').reset_index(drop=True)
ids = TESTE['event id'].tolist()
arquivos = sorted(f[:-4] for f in os.listdir(os.path.join(DADOS, 'test', 'S2')) if f.endswith('.tif'))
assert ids == arquivos, 'Os mapas do metadata não batem com os arquivos de test/S2. Cole esta saída no Claude.'
print(f'{len(ids)} mapas no teste, {TESTE.ems_code.nunique()} ativações do Copernicus EMS')

# Todas as cenas são anteriores à linha de base 04.00 do Sentinel-2 (25/01/2022), que acrescentou
# 1000 aos números digitais. Sem esse deslocamento, a escala é a mesma do Sen1Floods11.
datas_s2 = pd.to_datetime(TESTE['s2_date'], utc=True)
print('Cenas S2 de', datas_s2.min().date(), 'a', datas_s2.max().date())
assert (datas_s2 < pd.Timestamp('2022-01-25', tz='UTC')).all()

# Ordem das 13 bandas no ml4floods (BANDS_S2 em ml4floods/data/worldfloods/configs.py)
BANDAS_13 = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12']
SEIS = ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']        # as 6 bandas do Prithvi no Sen1Floods11
INDICES = [1, 2, 3, 8, 11, 12]
assert [BANDAS_13[i] for i in INDICES] == SEIS
assert [BANDAS_13[i] for i in [1, 2, 3, 7, 11, 12]][3] == 'B8'   # a configuração bgriswirs pega a B8: nunca usar

linhas = []
for eid in ids:
    caminho = lambda pasta, ext: os.path.join(DADOS, 'test', pasta, eid + ext)
    with rasterio.open(caminho('S2', '.tif')) as s:
        grade = (s.crs, s.transform, s.height, s.width)
        info = {'map_id': eid, 'bandas': s.count, 'dtype': s.dtypes[0], 'nodata': s.nodata,
                'descricoes': '|'.join(d or '' for d in s.descriptions), 'res_m': s.res[0], 'H': s.height, 'W': s.width}
        dec = max(1, max(s.height, s.width) // 1000)
        amostra = s.read(indexes=[i + 1 for i in INDICES], out_shape=(6, max(1, s.height // dec), max(1, s.width // dec)),
                         resampling=Resampling.nearest).astype(np.float32)
    validos = ~np.all(amostra == 0, axis=0)
    for b, nome in enumerate(SEIS):
        v = amostra[b][validos]
        info[nome + '_mediana'] = float(np.median(v)) if v.size else np.nan
    info['B2_p99'] = float(np.percentile(amostra[0][validos], 99)) if validos.any() else np.nan
    with rasterio.open(caminho('gt', '.tif')) as g:
        gt = g.read()
        info['gt_mesma_grade'] = (g.crs == grade[0]) and g.transform.almost_equals(grade[1]) and (g.height, g.width) == grade[2:]
    with rasterio.open(caminho('PERMANENTWATERJRC', '.tif')) as w:
        info['jrc_mesma_grade'] = (w.crs == grade[0]) and w.transform.almost_equals(grade[1]) and (w.height, w.width) == grade[2:]
    info['gt_canais'] = gt.shape[0]
    info['gt_nuvem_valores'] = ','.join(map(str, np.unique(gt[0])))
    info['gt_agua_valores'] = ','.join(map(str, np.unique(gt[1])))
    m = json.load(open(caminho('meta', '.json')))
    contagens = {'pixels invalid S2': int((gt[0] == 0).sum()), 'pixels clouds S2': int((gt[0] == 2).sum()),
                 'pixels water S2': int((gt[1] == 2).sum()), 'pixels land S2': int((gt[1] == 1).sum())}
    info['meta_confere_gt'] = all(m.get(k) == v for k, v in contagens.items())
    linhas.append(info)

INSP = pd.DataFrame(linhas)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print('\nDescrição das bandas no primeiro arquivo:', INSP.descricoes.iloc[0] or '(vazia)')
print('Chaves do meta JSON:', sorted(json.load(open(os.path.join(DADOS, 'test', 'meta', ids[0] + '.json'))).keys()))
print()
print(INSP[['map_id', 'bandas', 'dtype', 'nodata', 'res_m', 'H', 'W', 'gt_canais', 'gt_nuvem_valores', 'gt_agua_valores',
            'gt_mesma_grade', 'jrc_mesma_grade', 'meta_confere_gt']].to_string(index=False))
print('\nMediana por banda (números digitais, amostra decimada, pixels com dado):')
print(INSP[['map_id'] + [b + '_mediana' for b in SEIS] + ['B2_p99']].round(0).to_string(index=False))

problemas = []
if not (INSP.gt_canais == 2).all(): problemas.append('gt sem 2 canais')
if not INSP.gt_nuvem_valores.map(lambda s: set(s.split(',')) <= {'0', '1', '2'}).all(): problemas.append('valores inesperados no canal de nuvem')
if not INSP.gt_agua_valores.map(lambda s: set(s.split(',')) <= {'0', '1', '2'}).all(): problemas.append('valores inesperados no canal de água')
if not INSP.gt_mesma_grade.all(): problemas.append('gt em grade diferente da S2')
if not (INSP.bandas >= 13).all(): problemas.append('arquivo S2 com menos de 13 bandas')
print('\nProblemas:', problemas or 'nenhum')
INSP.to_csv(os.path.join(SAIDA, 't2_inspecao.csv'), index=False)
print('Fim da Parte A. Cole no Claude a saída das células 2 a 4.')

In [ ]:
# CÉLULA 4b. Conferir a ordem das bandas pela resolução nativa
# Os arquivos têm 15 bandas sem descrição. Banda de 20 m reamostrada para 10 m repete o valor em blocos de 2x2,
# e banda de 60 m em blocos de 6x6. A fração de vizinhos iguais separa as três resoluções e confirma que o
# índice 8 é uma banda de 20 m entre a B8 (10 m) e a B9 (60 m), isto é, a B8A.
ESPERADO = ['60', '10', '10', '10', '20', '20', '20', '10', '20', '60', '60', '20', '20']
classe = lambda f: '10' if f < 0.3 else ('20' if f < 0.7 else '60')
ORDEM = {}
for eid in ['EMSR286_08ITUANGONORTH_DEL_MONIT02_v1', 'EMSR347_06MWANZA_DEL_v1', 'EMSR466_AOI01_DEL_PRODUCT']:
    with rasterio.open(os.path.join(DADOS, 'test', 'S2', eid + '.tif')) as s:
        h, w = min(1024, s.height), min(1024, s.width)
        x = s.read(window=rasterio.windows.Window((s.width - w) // 2, (s.height - h) // 2, w, h))
    validos = ~np.all(x[:13] == 0, axis=0)
    ph, pv = validos[:, 1:] & validos[:, :-1], validos[1:, :] & validos[:-1, :]
    frac = [float((((b[:, 1:] == b[:, :-1]) & ph).sum() + ((b[1:, :] == b[:-1, :]) & pv).sum()) / max(ph.sum() + pv.sum(), 1))
            for b in x[:13]]
    ORDEM[eid] = [classe(f) for f in frac] == ESPERADO
    print(eid)
    print('  vizinhos iguais:', ' '.join(f'{BANDAS_13[i]}={f:.2f}' for i, f in enumerate(frac)))
    print('  banda 14, valores:', np.unique(x[13])[:8], '| banda 15, mín e máx:', int(x[14].min()), int(x[14].max()))
    print('  ordem confere com o ml4floods:', ORDEM[eid])
print('\nOrdem confirmada nos três mapas' if all(ORDEM.values()) else '\nOrdem NÃO confirmada. Cole esta saída no Claude.')

## Parte B

Só rode depois que o Claude confirmar o commit de `prereg/t2_criterios.md` com os limiares. Os valores da célula 6 têm de ser os mesmos do arquivo registrado.

Se a sessão do Colab caiu desde a Parte A, rode de novo as células 1 e 4 (a 4 lê do Drive quando o disco local foi apagado) e depois a 5 e a 6.

In [ ]:
# CÉLULA 5 (PARTE B). País de cada mapa e fração dos pixels rotulados em país do Sen1Floods11
!pip install -q geopandas
import geopandas as gpd
import urllib.request
from shapely.geometry import box
from rasterio import features
from rasterio.warp import transform_bounds
from rasterio.transform import array_bounds

NE_URLS = ['https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip',
           'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_admin_0_countries.geojson']
os.makedirs('/content/ne', exist_ok=True)
NE_ARQ = NE_URL = None
for u in NE_URLS:
    destino = os.path.join('/content/ne', os.path.basename(u))
    try:
        if not os.path.exists(destino):
            urllib.request.urlretrieve(u, destino)
        NE_ARQ, NE_URL = destino, u
        break
    except Exception as e:
        print('Falhou', u, e)
assert NE_ARQ, 'Não consegui baixar o Natural Earth. Cole esta saída no Claude.'
NE = gpd.read_file(NE_ARQ)
NE = NE.rename(columns={c: c.upper() for c in NE.columns if c != NE.geometry.name}).to_crs('EPSG:4326')
NE_SHA = sha256(NE_ARQ)
print(f'Natural Earth 1:10m: {NE_URL}  sha256 {NE_SHA[:16]}...  {len(NE)} unidades')

# Países com eventos no Sen1Floods11 (doc 22, seção 10.1). Mekong: os quatro países da bacia baixa, por precaução.
# Somália inclui a Somalilândia, que o Natural Earth separa.
TREINO = {'BOL': 'Bolivia', 'GHA': 'Ghana', 'IND': 'India', 'KHM': 'Cambodia', 'LAO': 'Laos', 'THA': 'Thailand',
          'VNM': 'Vietnam', 'NGA': 'Nigeria', 'PAK': 'Pakistan', 'PRY': 'Paraguay', 'SOM': 'Somalia',
          'SOL': 'Somaliland', 'ESP': 'Spain', 'LKA': 'Sri Lanka', 'USA': 'United States'}
ausentes = sorted(set(TREINO) - set(NE.ADM0_A3))
print('Códigos de treino ausentes no Natural Earth:', ausentes or 'nenhum')

PAIS = []
for eid in ids:
    with rasterio.open(os.path.join(DADOS, 'test', 'gt', eid + '.tif')) as g:
        gt = g.read(); crs, tr, H, W = g.crs, g.transform, g.height, g.width
    rotulado = gt[1] != 0
    n_rot = max(int(rotulado.sum()), 1)
    oeste, sul, leste, norte = transform_bounds(crs, 'EPSG:4326', *array_bounds(H, W, tr))
    caixa = box(oeste, sul, leste, norte)
    na_cena = NE[NE.intersects(caixa)]
    parte = {}
    for pais in na_cena.clip(caixa.buffer(0.05)).to_crs(crs).itertuples():
        if pais.geometry is None or pais.geometry.is_empty:
            continue
        msk = features.rasterize([(pais.geometry, 1)], out_shape=(H, W), transform=tr, fill=0, dtype='uint8').astype(bool)
        parte[(pais.ADMIN, pais.ADM0_A3)] = float((msk & rotulado).sum() / n_rot)
    frac = sum(v for (nome, cod), v in parte.items() if cod in TREINO)
    principal = max(parte, key=parte.get)[0] if parte and max(parte.values()) > 0 else 'none'
    PAIS.append({'map_id': eid, 'lat': (sul + norte) / 2, 'lon': (oeste + leste) / 2,
                 'country_main': principal,
                 'labeled_share_by_country': '|'.join(f'{nome}:{v:.4f}' for (nome, cod), v in sorted(parte.items(), key=lambda kv: -kv[1])),
                 'country_ems': json.load(open(os.path.join(DADOS, 'test', 'meta', eid + '.json'))).get('country'),
                 'training_countries_in_scene': '|'.join(sorted(na_cena[na_cena.ADM0_A3.isin(TREINO.keys())].ADMIN)),
                 'frac_labeled_in_training_country': frac,
                 'labeled_pixels': int(rotulado.sum()),
                 'clear_labeled_pixels': int((rotulado & (gt[0] == 1)).sum()),
                 'cloud_frac': float((gt[0] == 2).sum() / max(int((gt[0] != 0).sum()), 1))})
PAIS = pd.DataFrame(PAIS)
print(PAIS[['map_id', 'country_main', 'labeled_share_by_country', 'frac_labeled_in_training_country']]
      .round(4).to_string(index=False))

In [ ]:
# CÉLULA 6 (PARTE B). Aplicar os três critérios do doc 22, seção 10.2, e gravar a lista do T2
# Limiares registrados em prereg/t2_criterios.md antes desta célula. Não mudar depois de ver a saída.
MAX_FRAC_PAIS_TREINO = 0.05   # C1: no máximo 5% dos pixels rotulados em país do Sen1Floods11
NUVEM_MAX = 0.30              # C3: no máximo 30% de nuvem entre os pixels válidos do mapa

T = (TESTE.rename(columns={'event id': 'map_id', 'ems_code': 'event', 'aoi_code': 'aoi', 'satellite': 'ref_satellite',
                           'satellite date': 'ref_datetime', 's2_date': 's2_datetime'})
     [['map_id', 'event', 'aoi', 'ref_satellite', 'ref_datetime', 's2_datetime']]
     .merge(PAIS, on='map_id', validate='one_to_one'))
T['same_date'] = pd.to_datetime(T.ref_datetime, utc=True).dt.date == pd.to_datetime(T.s2_datetime, utc=True).dt.date
T['ref_is_s2'] = T.ref_satellite == 'Sentinel-2'
T['c1_country'] = T.frac_labeled_in_training_country <= MAX_FRAC_PAIS_TREINO
T['c2_aligned'] = T.same_date & T.ref_is_s2
T['c3_cloud'] = T.cloud_frac <= NUVEM_MAX
T['core'] = T.c1_country & T.c2_aligned & T.c3_cloud
T['full_set'] = T.c1_country & T.c3_cloud

def motivo(r):
    m = []
    if not r.c1_country: m.append('C1 training country')
    if not r.c2_aligned: m.append('C2 reference not same-day Sentinel-2')
    if not r.c3_cloud: m.append('C3 cloud')
    return '; '.join(m)
T['exclusion_reason'] = T.apply(motivo, axis=1)

outro_sensor = T[T.same_date & ~T.ref_is_s2]
print('Mesma data com outro sensor (fora do núcleo pela definição de C2):', list(outro_sensor.map_id) or 'nenhum')

ARQ = os.path.join(SAIDA, 't2_eventos.csv')
T.to_csv(ARQ, index=False, float_format='%.6f')
SHA_LISTA = sha256(ARQ)

print()
print(T[['map_id', 'event', 'country_main', 'ref_satellite', 'same_date', 'frac_labeled_in_training_country',
         'cloud_frac', 'clear_labeled_pixels', 'core', 'full_set', 'exclusion_reason']].round(3).to_string(index=False))
nuc = T[T.core]
print(f'\nNúcleo alinhado: {len(nuc)} mapas, {nuc.event.nunique()} eventos')
for ev, g in nuc.groupby('event'):
    print(f'  {ev} ({g.country_main.iloc[0]}): {", ".join(g.aoi)}  | {g.clear_labeled_pixels.sum():,} pixels rotulados sem nuvem')
comp = T[T.full_set]
print(f'Conjunto completo (C1 e C3, relatado à parte): {len(comp)} mapas, {comp.event.nunique()} eventos')

INFO = {'gerado_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        'dataset': REPO_WF, 'revisao': REVISAO_WF,
        'criterios': {'C1': f'frac_labeled_in_training_country <= {MAX_FRAC_PAIS_TREINO}',
                      'C2': 'reference map from Sentinel-2 on the same UTC date as the S2 scene',
                      'C3': f'cloud_frac <= {NUVEM_MAX} (cloud pixels over valid pixels of the gt cloud channel)'},
        'paises_treino_adm0_a3': sorted(TREINO), 'natural_earth': {'url': NE_URL, 'sha256': NE_SHA},
        'nucleo': {'mapas': list(nuc.map_id), 'eventos': sorted(nuc.event.unique())},
        'conjunto_completo': {'mapas': list(comp.map_id), 'eventos': sorted(comp.event.unique())},
        't2_eventos_csv_sha256': SHA_LISTA, 'versoes': VERSOES, 'modelo_rodado': False}
json.dump(INFO, open(os.path.join(SAIDA, 't2_eventos_info.json'), 'w'), indent=2, ensure_ascii=False)
print(f'\nt2_eventos.csv  sha256 {SHA_LISTA}')
print('Salvos em resultados/semana2: t2_eventos.csv e t2_eventos_info.json. Anexe os dois no Claude, com o t2_arquivos_sha256.csv.')